In [ ]:
# ── Notebook parameters ─────────────────────────────────────────
# Defaults for interactive use. extract_notebook_figures.py will
# override these via a cell injected right below this one.

NAME        = "SDS"
CHECKPOINT_DIR  = "../outputs/checkpoints/wandb-xxx"
CHECKPOINT_FILE = "step_xxx.pt"


# Concept Basis — Visualisation

Loads a checkpoint produced by `concept_basis.py` and visualises:

1. **Initial Fourier atoms** $\psi_{k_1,k_2,c}$ — the source dictionary.
2. **Learned (pushed-forward) atoms** $Q_\theta\psi_{k_1,k_2,c}$ — the concept basis.
3. **Random K-sparse combinations** $\sum_j c_j\,Q_\theta\psi_j$ — what the SDS loss actually sees during training.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, os
sys.path.insert(0, "..")

import torch
import numpy as np
import matplotlib.pyplot as plt

from notebook_helpers import field_to_img, to_rgb_image, render, visualize_generator_U


## 1. Load checkpoint

Point `CHECKPOINT_DIR` to a directory produced by `concept_basis.py`.
The directory must contain `config.yaml` and `best.pt` (or `latest.pt`).

In [ ]:
import yaml
from omegaconf import OmegaConf
from hydra.utils import instantiate
from infidictionary.neural_isometries import NeuralIsometry
from infidictionary.dictionaries import InfiDictionary
from infidictionary.domain_samplers import SquareSampler
from infidictionary.networks import NeuralField
from infidictionary.concept import CoefficientModel

# ── Point to a checkpoint ─────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

with open(f"{CHECKPOINT_DIR}/config.yaml") as f:
    conf = OmegaConf.create(yaml.safe_load(f))

neural_isometry: NeuralIsometry    = instantiate(conf.neural_isometry).to(device)
initial_dictionary: InfiDictionary = instantiate(conf.initial_dictionary)
variation_strength = float(conf.get("variation_strength", 1.0))

# Load mean_function if it was part of the training config
mean_function: NeuralField | None = None
if conf.get("mean_function") is not None:
    mean_function = instantiate(conf.mean_function).to(device)

# Load coefficient_model from config (_partial_: true → call with infidictionary)
coefficient_model: CoefficientModel | None = None
if conf.get("coefficient_model") is not None:
    coefficient_model = instantiate(conf.coefficient_model)(infidictionary=initial_dictionary)
    print(f"Loaded coefficient_model: {type(coefficient_model).__name__}")
else:
    print("Warning: no coefficient_model in config — combinations cell will use Gaussian fallback.")

ckpt = torch.load(f"{CHECKPOINT_DIR}/{CHECKPOINT_FILE}", map_location=device, weights_only=False)
neural_isometry.load_state_dict(ckpt["models"]["neural_isometry"])
if mean_function is not None and "mean_function" in ckpt.get("models", {}):
    mean_function.load_state_dict(ckpt["models"]["mean_function"])
    print("Loaded mean_function weights.")
elif mean_function is not None:
    print("Warning: mean_function in config but not found in checkpoint — using random init.")

model_state_kwargs = dict(conf.get("model_state_kwargs", {"num_steps": 20}))
pushforward_kwargs = dict(conf.get("pushforward_kwargs", {}))
neural_isometry.shuffle_model_state(**model_state_kwargs)
neural_isometry.eval()
if mean_function is not None:
    mean_function.eval()

print(f"Loaded : {CHECKPOINT_DIR}/{CHECKPOINT_FILE}")
print(f'  epoch  : {ckpt.get("epoch")}  |  metric : {ckpt.get("metric", float("nan")):.4f}')
print(f"  model_state_kwargs : {model_state_kwargs}")
print(f"  variation_strength : {variation_strength}")


## 2. Evaluate atoms on a dense grid

Evaluate both the initial Fourier atoms and their pushforward on a `N_PER_DIM × N_PER_DIM`
regular grid.  `NUM_TRUNCATED` is the L∞ half-bandwidth; it gives
$(2 \cdot \text{NUM\_TRUNCATED}+1)^2 \times 3$ atoms in total.

In [ ]:
NUM_TRUNCATED = 4   # L∞ half-bandwidth → (2·3+1)² = 49 spatial atoms × 3 channels = 147 total
N_PER_DIM     = 64

indices = initial_dictionary.get_truncated_indices(num_truncated=NUM_TRUNCATED).to(device)
print(f"Indices : {indices.shape}  "
      f"(spatial {indices[:, :-1].min()}..{indices[:, :-1].max()}, "
      f"channels {indices[:, -1].unique().tolist()})")

# SquareSampler(stratified=True, add_noise=False) produces an exact regular grid
# with indexing='ij': coords[i*n+j] = (x_i, y_j), i.e. x varies along the slow
# (outer) axis.  When reshaping to (n, n, C) the first axis is x (→ rows of the
# image array) and the second is y (→ columns).  We flip axis-0 in atom_as_img
# so that increasing y points upward in imshow, matching a standard plot.
grid_coords    = SquareSampler(stratified=True, add_noise=False).sample(N_PER_DIM).to(device)
grid_logabsdet = torch.zeros(grid_coords.shape[0], device=device)

with torch.no_grad():
    grid_atoms = initial_dictionary.get_atoms(grid_coords, indices)   # (A, N, 3)
    _, _, grid_deformed = neural_isometry.pushforward(
        src_coords=grid_coords,
        src_logabsdet=grid_logabsdet,
        src_field=grid_atoms,
        **pushforward_kwargs,
    )   # (A, N, 3)

    # Evaluate the mean function on the same grid if available.
    # Apply tanh to match the training loop: concept_basis.py applies tanh as an
    # output activation before passing to the concept loss (prevents dead zones
    # from clamp and bounds output to (-1, 1) regardless of raw network values).
    if mean_function is not None:
        mean_vals = mean_function(grid_coords).tanh()   # (N, C), in (-1, 1)
        mean_img_np = mean_vals.reshape(N_PER_DIM, N_PER_DIM, -1).cpu().numpy()
    else:
        mean_img_np = None

grid_atoms_np    = grid_atoms.cpu().numpy()    # (A, N, 3)
grid_deformed_np = grid_deformed.cpu().numpy() # (A, N, 3)
print(f"grid_atoms:    {grid_atoms_np.shape}")
print(f"grid_deformed: {grid_deformed_np.shape}")
if mean_img_np is not None:
    print(f"mean_img:      {mean_img_np.shape}  "
          f"range [{mean_img_np.min():.3f}, {mean_img_np.max():.3f}]")

## 3. Triptych visualisation

Each panel shows one RGB channel.  Rows = $k_1$ values, columns = $k_2$ values.
Values are clamped to $[-1,1]$ and rescaled to $[0,1]$ for display.

In [ ]:
idx_np  = indices.cpu().numpy()   # (A, 3)
k1_vals = sorted(set(idx_np[:, 0].tolist()))
k2_vals = sorted(set(idx_np[:, 1].tolist()))
ch_names_rgb = ["R", "G", "B"]

lookup = {(int(idx_np[i, 0]), int(idx_np[i, 1]), int(idx_np[i, 2])): i
          for i in range(len(idx_np))}


def atom_as_img(atoms_arr: np.ndarray, i: int, n: int) -> np.ndarray:
    """Return atom i as an (n, n, 3) RGB float array ready for imshow.

    Each channel is normalized independently to [0, 1].  For near-constant
    channels (range < 1e-6) the fallback is the channel mean scaled to the
    global range, so DC atoms show their channel colour rather than grey.
    """
    raw = atoms_arr[i].reshape(n, n, 3)
    lo  = raw.min(axis=(0, 1), keepdims=True)
    hi  = raw.max(axis=(0, 1), keepdims=True)
    rng = hi - lo
    g_lo, g_hi = raw.min(), raw.max()
    mean = raw.mean(axis=(0, 1), keepdims=True)
    if g_hi - g_lo > 1e-6:
        fallback = np.broadcast_to(np.clip((mean - g_lo) / (g_hi - g_lo), 0.0, 1.0), raw.shape).copy()
    else:
        fallback = np.full_like(raw, 0.1)
    processed = np.divide(raw - lo, rng, out=fallback, where=rng > 1e-6)
    return np.flip(processed, axis=0)


def atom_as_gray(atoms_arr: np.ndarray, i: int, n: int) -> np.ndarray:
    """Return atom i as an (n, n) grayscale float array (channel average)."""
    raw  = atoms_arr[i].reshape(n, n, 3)
    gray = raw.mean(axis=-1)
    lo, hi = gray.min(), gray.max()
    tile = np.divide(gray - lo, hi - lo,
                     out=np.full_like(gray, 0.5),
                     where=(hi - lo) > 1e-6)
    return np.flip(tile, axis=0)


def make_mosaic(atoms_arr: np.ndarray, channel: int, n: int) -> np.ndarray:
    """RGB mosaic for atoms whose multi-index ends with `channel`."""
    n_k1, n_k2 = len(k1_vals), len(k2_vals)
    mosaic = np.zeros((n_k1 * n, n_k2 * n, 3))
    for ri, k1 in enumerate(reversed(k1_vals)):
        for ci, k2 in enumerate(k2_vals):
            key = (int(k1), int(k2), channel)
            if key in lookup:
                mosaic[ri*n:(ri+1)*n, ci*n:(ci+1)*n] = atom_as_img(atoms_arr, lookup[key], n)
    return mosaic


def make_gray_mosaic(atoms_arr: np.ndarray, channel: int, n: int) -> np.ndarray:
    """Grayscale mosaic (channel average) for atoms whose multi-index ends with `channel`."""
    n_k1, n_k2 = len(k1_vals), len(k2_vals)
    mosaic = np.full((n_k1 * n, n_k2 * n), 0.5)
    for ri, k1 in enumerate(reversed(k1_vals)):
        for ci, k2 in enumerate(k2_vals):
            key = (int(k1), int(k2), channel)
            if key in lookup:
                mosaic[ri*n:(ri+1)*n, ci*n:(ci+1)*n] = atom_as_gray(atoms_arr, lookup[key], n)
    return mosaic


def _add_axis_labels(ax, n):
    k1_rev = list(reversed(k1_vals))
    ax.set_xticks([ci * n + n // 2 for ci in range(len(k2_vals))])
    ax.set_xticklabels([str(int(k)) for k in k2_vals], fontsize=7)
    ax.set_xlabel("$k_2$")
    ax.set_yticks([ri * n + n // 2 for ri in range(len(k1_vals))])
    ax.set_yticklabels([str(int(k)) for k in k1_rev], fontsize=7)
    ax.set_ylabel("$k_1$")


def plot_triptych(atoms_arr: np.ndarray, title: str, n: int) -> None:
    """RGB triptych: one panel per channel group (c=0,1,2)."""
    fig, axes = plt.subplots(1, 3, figsize=(21, 7))
    for c, (ax, ch) in enumerate(zip(axes, ch_names_rgb)):
        ax.imshow(make_mosaic(atoms_arr, c, n), origin="lower")
        ax.set_title(f"Channel {ch}", fontsize=11)
        _add_axis_labels(ax, n)
    fig.suptitle(title, fontsize=13)
    plt.tight_layout()
    plt.show()


def plot_gray_triptych(atoms_arr: np.ndarray, title: str, n: int) -> None:
    """Grayscale triptych (channel average): one panel per channel group."""
    fig, axes = plt.subplots(1, 3, figsize=(21, 7))
    for c, (ax, ch) in enumerate(zip(axes, ch_names_rgb)):
        ax.imshow(make_gray_mosaic(atoms_arr, c, n), origin="lower", cmap="gray", vmin=0, vmax=1)
        ax.set_title(f"Channel {ch}  ($c={c}$)", fontsize=11)
        _add_axis_labels(ax, n)
    fig.suptitle(title, fontsize=13)
    plt.tight_layout()
    plt.show()


plot_triptych(grid_atoms_np,    "Initial Fourier atoms  $\\psi_{k_1,k_2,c}$",                              N_PER_DIM)
plot_triptych(grid_deformed_np, "Learned concept basis  $Q_\\theta\\psi_{k_1,k_2,c}$",                     N_PER_DIM)
plot_gray_triptych(grid_deformed_np, "Learned concept basis — grayscale  $Q_\\theta\\psi_{k_1,k_2,c}$",    N_PER_DIM)

# ── Mean function ─────────────────────────────────────────────────────────────
if mean_img_np is not None:
    mean_display = (np.clip(np.flip(mean_img_np, axis=0), -1, 1) + 1) / 2

    fig, axes = plt.subplots(1, 4, figsize=(18, 4))
    axes[0].imshow(mean_display, origin="lower")
    axes[0].set_title("RGB composite  $\\mu_\\phi(x)$", fontsize=11)
    axes[0].axis("off")
    for c, (ax, ch) in enumerate(zip(axes[1:], ch_names_rgb)):
        ax.imshow(mean_display[:, :, c], origin="lower", cmap="RdBu_r", vmin=0, vmax=1)
        ax.set_title(f"Channel {ch}", fontsize=11)
        ax.axis("off")
    fig.suptitle(
        "Learned mean function  $\\mu_\\phi$  "
        f"(range [{mean_img_np.min():.2f}, {mean_img_np.max():.2f}])",
        fontsize=13,
    )
    plt.tight_layout()
    plt.show()

## 4. Isometry check

Verify that $Q_\theta$ preserves the $L^2$ inner product by computing the Gram matrix
$$G_{ab} = \langle Q_\theta\psi_a,\, Q_\theta\psi_b \rangle = \frac{1}{N}\sum_n e^{\lambda_n}\,(Q_\theta\psi_a)(x_n)\cdot(Q_\theta\psi_b)(x_n)$$
and comparing to $I$.  The residual $\|G - I\|_F$ on the initial atoms is purely quadrature error; the same quantity on the deformed atoms also includes any isometry violation introduced by the numerical integrator.

In [ ]:
from infidictionary.utils import pairwise_inner_product

with torch.no_grad():
    gram_initial  = pairwise_inner_product(grid_atoms,    grid_atoms,    grid_logabsdet)  # (A, A)
    gram_deformed = pairwise_inner_product(grid_deformed, grid_deformed, grid_logabsdet)  # (A, A)

A = gram_initial.shape[0]
eye = torch.eye(A, device=gram_initial.device)

err_initial  = (gram_initial  - eye).norm().item()
err_deformed = (gram_deformed - eye).norm().item()

print(f"{'':28s}  {'||G − I||_F':>12s}  {'per atom (/ √A)':>16s}  {'max |G−I|':>10s}")
print(f"{'Initial atoms':28s}  {err_initial:>12.4f}  {err_initial / A**0.5:>16.4f}  "
      f"{(gram_initial - eye).abs().max().item():>10.4f}")
print(f"{'Deformed atoms  (Q_θ ψ)':28s}  {err_deformed:>12.4f}  {err_deformed / A**0.5:>16.4f}  "
      f"{(gram_deformed - eye).abs().max().item():>10.4f}")

# ── Heatmap of the two Gram matrices ─────────────────────────────────────────
gram_i_np = gram_initial.cpu().numpy()
gram_d_np = gram_deformed.cpu().numpy()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

vmax_i = np.abs(gram_i_np).max()
im0 = axes[0].imshow(gram_i_np, cmap="RdBu_r", vmin=-vmax_i, vmax=vmax_i)
axes[0].set_title(r"$G_{\rm initial}$  (quadrature error only)", fontsize=11)
plt.colorbar(im0, ax=axes[0], fraction=0.046)

vmax_d = np.abs(gram_d_np).max()
im1 = axes[1].imshow(gram_d_np, cmap="RdBu_r", vmin=-vmax_d, vmax=vmax_d)
axes[1].set_title(r"$G_{\rm deformed}$  ($Q_\theta\psi$ Gram matrix)", fontsize=11)
plt.colorbar(im1, ax=axes[1], fraction=0.046)

err_map = (gram_d_np - np.eye(A))
vmax_e = np.abs(err_map).max()
im2 = axes[2].imshow(err_map, cmap="RdBu_r", vmin=-vmax_e, vmax=vmax_e)
axes[2].set_title(r"$G_{\rm deformed} - I$  (isometry residual)", fontsize=11)
plt.colorbar(im2, ax=axes[2], fraction=0.046)

for ax in axes:
    ax.set_xlabel("atom index $b$")
    ax.set_ylabel("atom index $a$")

fig.suptitle(
    rf"Isometry check  —  $A={A}$ atoms,  $N={grid_coords.shape[0]}$ quadrature pts",
    fontsize=13,
)
plt.tight_layout()
plt.show()

## 4. Random K-sparse combinations

Sample several random sets of K atom indices and decayed coefficients
— exactly as done in training — and render $\sum_j c_j\,Q_\theta\psi_j$.
If a mean function was trained, it is added to each combination before display,
showing the full image $\mu_\phi(x) + \sum_j c_j\,Q_\theta\psi_j$ as the concept loss sees it.

In [ ]:
from infidictionary.concept import GaussianCoefficientModel

N_COMBINATIONS = 12   # number of random combinations to display
N_COLS         = 4

# ── Build index set (mirrors training: exact high-prob stratum + MC tail) ─────
tail_probability = float(conf.get("tail_probability", 1e-3))
num_tail_samples = int(conf.get("num_tail_samples", 64))

idx_exact    = initial_dictionary.get_high_probability_indices(tail_probability).to(device)
idx_tail_raw = initial_dictionary.sample_indices(num_tail_samples).to(device)
in_exact     = (idx_tail_raw[:, None, :] == idx_exact[None, :, :]).all(dim=-1).any(dim=-1)
idx_tail     = torch.unique(idx_tail_raw[~in_exact], dim=0)
indices_vis  = torch.cat([idx_exact, idx_tail], dim=0)            # (A, d+1)
pmfs_vis     = initial_dictionary.get_index_pmfs(indices_vis).to(device)  # (A,)
A_vis        = indices_vis.shape[0]
n_exact      = idx_exact.shape[0]

# ── Coefficient model: use loaded model or fall back to Gaussian ──────────────
coeff_model_vis = coefficient_model if coefficient_model is not None else GaussianCoefficientModel(
    infidictionary=initial_dictionary,
    truncation=4,
    coefficient_decay=0.5,
)

# ── Evaluate initial atoms and push forward once ──────────────────────────────
with torch.no_grad():
    init_atoms_vis = initial_dictionary.get_atoms(grid_coords, indices_vis)  # (A, N, C)
    _, _, pushed_vis = neural_isometry.pushforward(
        src_coords=grid_coords,
        src_logabsdet=grid_logabsdet,
        src_field=init_atoms_vis,
        **pushforward_kwargs,
    )  # (A, N, C)

    # Sample N_COMBINATIONS coefficient vectors from the prior.
    coeffs_all = coeff_model_vis.sample(indices_vis, N_COMBINATIONS, device)  # (B, A)

    combos = torch.einsum("ba,anc->bnc", coeffs_all, pushed_vis)            # (B, N, C)

    # Add mean_function if available (affine subspace: tanh(mean) + variation).
    if mean_function is not None:
        mean_vals_vis = mean_function(grid_coords).tanh()                  # (N, C), in (-1, 1)
        combos = combos * variation_strength + mean_vals_vis.unsqueeze(0)  # (B, N, C)

    # Apply tanh to the full combined image — matches training exactly.
    combos = combos.tanh()                                                 # (B, N, C), in (-1, 1)

C_vis      = combos.shape[-1]
combos_np  = combos.reshape(N_COMBINATIONS, N_PER_DIM, N_PER_DIM, C_vis).cpu().numpy()
combos_np  = (np.flip(combos_np, axis=1) + 1) / 2
coeffs_np  = coeffs_all.cpu().numpy()   # (B, A)
pmfs_np    = pmfs_vis.cpu().numpy()     # (A,)

has_mean     = mean_function is not None
title_suffix = r"$+\;\mu_\phi$  " if has_mean else ""
model_name   = type(coeff_model_vis).__name__

# ── Figure 1: rendered combination images ────────────────────────────────────
n_rows = (N_COMBINATIONS + N_COLS - 1) // N_COLS
fig, axes = plt.subplots(n_rows, N_COLS, figsize=(N_COLS * 3, n_rows * 3))
axes = axes.flatten()
for i, (ax, img) in enumerate(zip(axes, combos_np)):
    ax.imshow(np.clip(img, 0, 1), origin="lower")
    ax.set_title(f"#{i}", fontsize=9)
    ax.axis("off")
for ax in axes[N_COMBINATIONS:]:
    ax.axis("off")
fig.suptitle(
    rf"Random combinations  $\tanh(\sum_j c_j\,Q_\theta\psi_j)$  {title_suffix}"
    f"[{model_name}, A={A_vis} atoms]",
    fontsize=12,
)
plt.tight_layout()
plt.show()

# ── Figure 2: coefficient bar charts (one row per combination) ────────────────
sort_order    = np.argsort(-pmfs_np)            # (A,) descending PMF
coeffs_sorted = coeffs_np[:, sort_order]        # (B, A)
pmfs_sorted   = pmfs_np[sort_order]             # (A,)
atom_x        = np.arange(A_vis)

vmax = np.abs(coeffs_sorted).max()

fig, axes = plt.subplots(
    N_COMBINATIONS + 1, 1,
    figsize=(max(8, A_vis // 8), (N_COMBINATIONS + 1) * 1.5),
    sharex=True,
    gridspec_kw={"height_ratios": [1.5] + [1.0] * N_COMBINATIONS},
)

ax_pmf = axes[0]
pmf_vals = np.where(pmfs_sorted > 0, pmfs_sorted, np.nan)
ax_pmf.bar(atom_x, pmf_vals, width=1.0, color="gray", alpha=0.6)
ax_pmf.set_yscale("log")
ax_pmf.set_ylabel("PMF", fontsize=7, color="gray")
ax_pmf.tick_params(axis="y", labelsize=6, colors="gray")
ax_pmf.axvspan(n_exact - 0.5, A_vis - 0.5, alpha=0.10, color="orange")
ax_pmf.spines[["top", "right"]].set_visible(False)
ax_pmf.set_title("Prior PMF (log scale)", fontsize=8, color="gray", pad=2)

for i, ax in enumerate(axes[1:]):
    colors = np.where(coeffs_sorted[i] >= 0, "steelblue", "tomato")
    ax.bar(atom_x, coeffs_sorted[i], width=1.0, color=colors, alpha=0.85)
    ax.axhline(0, color="black", linewidth=0.4, alpha=0.5)
    ax.axvspan(n_exact - 0.5, A_vis - 0.5, alpha=0.06, color="orange")
    ax.set_ylim(-vmax * 1.15, vmax * 1.15)
    ax.set_ylabel(f"#{i}", fontsize=7, rotation=0, labelpad=18)
    ax.tick_params(axis="y", labelsize=6)
    ax.spines[["top", "right"]].set_visible(False)

axes[-1].set_xlabel("Atom index  (sorted by PMF ↓,  orange = tail)", fontsize=9)

fig.suptitle(
    f"Sampled coefficients per combination  [{model_name}]  —  "
    f"exact: {n_exact} atoms  |  tail: {A_vis - n_exact} atoms",
    fontsize=11, y=1.005,
)
plt.tight_layout()
plt.show()


## 5. Generator field $U$ and rotation $K_R$ at different timesteps

Visualise the $R$ columns of the generator field $U_\theta(t, x) \in \mathbb{R}^{R \times C}$ and the $R \times R$ orthogonal matrix $\exp(K_R - K_R^\top)$ that together define the skew-adjoint operator $\mathcal{K}_\theta(t) = U_\theta(t)\,(K_R - K_R^\top)\,U_\theta(t)^\ast$ at several timesteps. Rows $0..R-1$ show the spatial columns $u_r$; the bottom row shows the finite rotation.

In [ ]:
from infidictionary.neural_isometries.eulerian import EulerianIsometry

if not isinstance(neural_isometry, EulerianIsometry):
    raise TypeError(
        f"Generator field visualisation requires EulerianIsometry, "
        f"got {type(neural_isometry).__name__}"
    )

# ── Settings ──────────────────────────────────────────────────────────────────
TEF_DENSITY = 128          # grid resolution: DENSITY × DENSITY points
N_TIMESTEPS = 9

timesteps = neural_isometry.tspan.clone()
# pick a subsequence of size N_TIMESTEPS from timesteps at random
timesteps = timesteps[torch.randperm(len(timesteps))[:N_TIMESTEPS]]
# sort
timesteps = torch.sort(timesteps)[0]

# ── Evaluate ──────────────────────────────────────────────────────────────────
tef_coords = SquareSampler(stratified=True, add_noise=False).sample(TEF_DENSITY).to(device)
# SquareSampler uses indexing='ij': coords[i·N+j] = (linspace[i], linspace[j]).
# field_to_img reshapes flat → (N,N) preserving that order, so imshow row i
# corresponds to the first coord and col j to the second — transposing x and y.
# Swapping columns converts ij-layout to xy-layout so the display is correct.
tef_coords = tef_coords[:, [1, 0]]
tef_coords[:, 0] = 1 - tef_coords[:, 0]

visualize_generator_U(neural_isometry, tef_coords, TEF_DENSITY, timesteps, device)
